# 편의점 로봇 추천 시스템 ① Instacart 데이터 분석과 lift 계산

**목표:** 실제 장보기 주문 데이터(Instacart)로 "같이 사는 상품"을 찾아, 웹의 장바구니 추천에 쓸 점수 파일을 만든다.

**지금까지 한 것**
1. Instacart 데이터 다운로드 (주문 약 321만 건, 상품 49,688개, 진열 코너 134개)
2. 우리 상품 22종을 진열 코너(aisle)에 연결
3. 주문마다 코너를 세서 동시 구매 횟수 c(A, B)와 lift 계산
4. 큰 장바구니 편향 확인 → 편의점에 맞게 **코너 2~5개 주문만** 사용하기로 결정
5. 웹 서버용 `rec_scores.json`, 우리 상품용 `our_recs.json` 저장

> 자세한 분석 내용은 함께 만든 `recommender_report.html` 참고

## 0. 설정

In [1]:
import os, json
import numpy as np, pandas as pd
from scipy import sparse

from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/2026_02_ROKEY/07_cobot2"
DATA = f"{BASE}/instacart"        # 원본 CSV
OUT  = f"{BASE}/recommender"      # 결과 파일
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.max_rows", 200)

Mounted at /content/drive


## 1. 데이터 다운로드 (처음 한 번만)
Drive에 CSV가 없을 때만 Kaggle에서 받는다. 코랩 왼쪽 🔑 Secrets에 `KAGGLE_USERNAME`, `KAGGLE_KEY`를 등록해 둘 것.

In [2]:
if not os.path.exists(f"{DATA}/order_products__prior.csv"):
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    !pip -q install kaggle
    !kaggle datasets download -d psparks/instacart-market-basket-analysis -p /content/instacart --unzip
    !cp /content/instacart/*.csv "{DATA}/"
!ls -lh "{DATA}"

total 681M
-rw------- 1 root root 2.6K Nov 26 05:24 aisles.csv
-rw------- 1 root root  270 Nov 26 05:24 departments.csv
-rw------- 1 root root 551M Nov 26 05:24 order_products__prior.csv
-rw------- 1 root root  24M Nov 26 05:24 order_products__train.csv
-rw------- 1 root root 104M Nov 26 05:24 orders.csv
-rw------- 1 root root 2.1M Nov 26 05:24 products.csv


## 2. 데이터 둘러보기
| 파일 | 한 줄의 의미 |
|---|---|
| `departments.csv` | 큰 분류 (21개) |
| `aisles.csv` | 진열 코너 (134개) |
| `products.csv` | 상품 (49,688개) → 코너, 큰 분류 |
| `orders.csv` | 주문 1건 (누가, 몇 번째, 요일, 시간) |
| `order_products__prior.csv` | 과거 주문의 상품 1개 → **학습용** |
| `order_products__train.csv` | 손님별 마지막 주문의 상품 1개 → **평가용** (여기서는 사용 안 함) |

In [3]:
departments = pd.read_csv(f"{DATA}/departments.csv")
aisles      = pd.read_csv(f"{DATA}/aisles.csv")
products    = pd.read_csv(f"{DATA}/products.csv")
orders      = pd.read_csv(f"{DATA}/orders.csv")
prior       = pd.read_csv(f"{DATA}/order_products__prior.csv",
                          dtype={"order_id": "int32", "product_id": "int32",
                                 "add_to_cart_order": "int16", "reordered": "int8"})

aisle_name = dict(zip(aisles.aisle_id, aisles.aisle))
print(orders.eval_set.value_counts())
print("prior 상품 줄 수:", len(prior))

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64
prior 상품 줄 수: 32434489


In [4]:
# 큰 분류(department) > 코너(aisle) 구조
tree = (products.merge(aisles, on="aisle_id").merge(departments, on="department_id")
                .groupby("department")["aisle"].unique())
for dep, ais in tree.items():
    print(f"{dep}: {', '.join(sorted(ais))}")

alcohol: beers coolers, red wines, specialty wines champagnes, spirits, white wines
babies: baby accessories, baby bath body care, baby food formula, diapers wipes
bakery: bakery desserts, bread, breakfast bakery, buns rolls, tortillas flat bread
beverages: cocoa drink mixes, coffee, energy sports drinks, juice nectars, refrigerated, soft drinks, tea, water seltzer sparkling water
breakfast: breakfast bars pastries, cereal, granola, hot cereal pancake mixes
bulk: bulk dried fruits vegetables, bulk grains rice dried goods
canned goods: canned fruit applesauce, canned jarred vegetables, canned meals beans, canned meat seafood, soup broth bouillon
dairy eggs: butter, cream, eggs, milk, other creams cheeses, packaged cheese, refrigerated pudding desserts, soy lactosefree, specialty cheeses, yogurt
deli: fresh dips tapenades, lunch meat, prepared meals, prepared soups salads, tofu meat alternatives
dry goods pasta: dry pasta, fresh pasta, grains rice dried goods, instant foods, pasta sauce


In [5]:
# 예시: 손님 1의 주문 기록
u = orders[orders.user_id == 1].sort_values("order_number")
display(u)

detail = (prior[prior.order_id.isin(u.order_id)]
          .merge(products, on="product_id").merge(aisles, on="aisle_id")
          .merge(u[["order_id", "order_number"]], on="order_id")
          .sort_values(["order_number", "add_to_cart_order"]))
display(detail[["order_number", "add_to_cart_order", "product_name", "aisle", "reordered"]].head(20))

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0
5,3367565,1,prior,6,2,7,19.0
6,550135,1,prior,7,1,9,20.0
7,3108588,1,prior,8,1,14,14.0
8,2295261,1,prior,9,1,16,0.0
9,2550362,1,prior,10,4,8,30.0


,order_number,add_to_cart_order,product_name,aisle,reordered
35,1,1,Soda,soft drinks,0
36,1,2,Organic Unsweetened Vanilla Almond Milk,soy lactosefree,0
37,1,3,Original Beef Jerky,popcorn jerky,0
38,1,4,Aged White Cheddar Popcorn,popcorn jerky,0
39,1,5,XL Pick-A-Size Paper Towel Rolls,paper goods,0
29,2,1,Soda,soft drinks,1
30,2,2,Pistachios,nuts seeds dried fruit,0
31,2,3,Original Beef Jerky,popcorn jerky,1
32,2,4,Bag of Organic Bananas,fresh fruits,0
33,2,5,Aged White Cheddar Popcorn,popcorn jerky,1


## 3. 우리 상품 ↔ 진열 코너 연결
- 실물 9종(로봇이 담음) 중 신규 실험용 상품은 연결하지 않음 (콜드 스타트 실험용)
- 참치캔과 스팸은 같은 코너(95) → 추천 점수가 같으므로 서로 **대체 상품**으로 사용
- 맥주 등 주류는 미성년자 문제로 제외

In [6]:
OURS = pd.DataFrame([
    # id,          이름,        aisle_id, 로봇
    ("tuna_can",    "참치캔",      95, True),
    ("spam_can",    "스팸 통조림",  95, True),
    ("milk_pack",   "소형 우유팩",  84, True),
    ("soap_box",    "비누 상자",    25, True),
    ("choco_box",   "초콜릿 상자",  45, True),
    ("snack_bag",   "과자 봉지",   107, True),
    ("mini_tissue", "미니 티슈",    54, True),
    ("egg_pack",    "계란 모형",    86, True),
    ("cup_ramen",   "컵라면",       4, False),
    ("cola",        "콜라",        77, False),
    ("water",       "생수",       115, False),
    ("can_coffee",  "캔커피",      26, False),
    ("yogurt",      "요구르트",    120, False),
    ("bread",       "식빵",       112, False),
    ("cookie",      "쿠키",        61, False),
    ("ice_cream",   "아이스크림",   37, False),
    ("cereal",      "시리얼",     121, False),
    ("nuts",        "견과류",     117, False),
    ("toothpaste",  "치약",        20, False),
    ("juice",       "주스",        98, False),
    ("sausage",     "소시지",     106, False),
    ("cracker",     "크래커",      78, False),
    ("lunchbox",    "도시락",      13, False),
], columns=["id", "name", "aisle_id", "robot"])

info = (aisles.merge(products[["aisle_id", "department_id"]].drop_duplicates(), on="aisle_id")
              .merge(departments, on="department_id"))
OURS = OURS.merge(info, on="aisle_id", how="left")
display(OURS[["id", "name", "robot", "aisle_id", "aisle", "department"]])

,id,name,robot,aisle_id,aisle,department
0,tuna_can,참치캔,True,95,canned meat seafood,canned goods
1,spam_can,스팸 통조림,True,95,canned meat seafood,canned goods
2,milk_pack,소형 우유팩,True,84,milk,dairy eggs
3,soap_box,비누 상자,True,25,soap,personal care
4,choco_box,초콜릿 상자,True,45,candy chocolate,snacks
5,snack_bag,과자 봉지,True,107,chips pretzels,snacks
6,mini_tissue,미니 티슈,True,54,paper goods,household
7,egg_pack,계란 모형,True,86,eggs,dairy eggs
8,cup_ramen,컵라면,False,4,instant foods,dry goods pasta
9,cola,콜라,False,77,soft drinks,beverages


## 4. 코너 동시 구매 횟수와 lift
1. 주문마다 코너 집합을 만든다 (같은 코너는 한 번만, `missing`·`other` 제외)
2. 주문 × 코너 0/1 행렬 X를 만들고 `C = Xᵀ X` 로 한 번에 계산
   - `C[a, a] = c(a)` : 코너 a가 나온 주문 수
   - `C[a, b] = c(a, b)` : 코너 a, b가 같이 나온 주문 수
3. **lift(a, b) = c(a, b) × N / (c(a) × c(b))** → 1보다 크면 우연보다 더 같이 산다

In [7]:
basket = (prior[["order_id", "product_id"]]
          .merge(products[["product_id", "aisle_id"]], on="product_id")[["order_id", "aisle_id"]]
          .drop_duplicates())
basket = basket[~basket.aisle_id.isin([6, 100])]          # other, missing 제외

aisle_ids = sorted(basket.aisle_id.unique())
a2i = {a: i for i, a in enumerate(aisle_ids)}
rows, _ = pd.factorize(basket.order_id)
X = sparse.csr_matrix((np.ones(len(basket), dtype=np.int64),
                       (rows, basket.aisle_id.map(a2i).values)))
sizes = np.asarray(X.sum(axis=1)).ravel()                 # 주문별 코너 개수
print("주문 수:", X.shape[0], " 코너 수:", len(aisle_ids))
print(pd.Series(sizes).describe())

주문 수: 3214236  코너 수: 132
count    3.214236e+06
mean     7.231554e+00
std      4.744003e+00
min      1.000000e+00
25%      4.000000e+00
50%      6.000000e+00
75%      1.000000e+01
max      5.700000e+01
dtype: float64


In [8]:
def build(mask):
    """mask에 해당하는 주문만으로 C(동시 구매 행렬), lift, 주문 수 계산"""
    Xs = X[mask]
    Ns = Xs.shape[0]
    Cs = (Xs.T @ Xs).toarray()
    cs = np.diag(Cs).astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        L = np.nan_to_num(Cs * Ns / np.outer(cs, cs))
    np.fill_diagonal(L, 0)
    return Cs, L, Ns

VERSIONS = {
    "전체 주문":   (np.ones_like(sizes, dtype=bool),       1000),
    "코너 2~8개": ((sizes >= 2) & (sizes <= 8),            300),
    "코너 2~5개": ((sizes >= 2) & (sizes <= 5),            200),
}
results = {k: build(m) + (mc,) for k, (m, mc) in VERSIONS.items()}
for k, (Cm, L, n, mc) in results.items():
    print(f"{k:8s} 주문 수 {n:>9,}  (최소 동시 구매 {mc})")

전체 주문    주문 수 3,214,236  (최소 동시 구매 1000)
코너 2~8개  주문 수 1,965,126  (최소 동시 구매 300)
코너 2~5개  주문 수 1,167,496  (최소 동시 구매 200)


In [9]:
# 전체 코너 기준 lift 상위 (전체 주문)
C_all, L_all, N_all, _ = results["전체 주문"]
def top_all(aisle, k=5, min_count=1000):
    i = a2i[aisles.set_index("aisle").aisle_id[aisle]]
    js = [j for j in np.argsort(-L_all[i]) if C_all[i, j] >= min_count and j != i][:k]
    print(f"[{aisle}] 인기도={C_all[i, i] / N_all:.3f}")
    for j in js:
        print(f"  {aisle_name[aisle_ids[j]]:28s} lift={L_all[i, j]:.2f}  count={C_all[i, j]}")

for a in ["canned meat seafood", "chips pretzels", "milk", "paper goods"]:
    top_all(a)

[canned meat seafood] 인기도=0.019
  packaged seafood             lift=3.12  count=1223
  pickled goods olives         lift=2.64  count=5144
  cat food care                lift=2.62  count=1731
  condiments                   lift=2.46  count=8993
  body lotions soap            lift=2.06  count=1398
[chips pretzels] 인기도=0.167
  preserved dips spreads       lift=2.56  count=43727
  fresh dips tapenades         lift=2.14  count=112549
  cookies cakes                lift=2.05  count=65151
  trail mix snack mix          lift=2.05  count=11725
  fruit vegetable snacks       lift=2.02  count=50280
[milk] 인기도=0.245
  baby food formula            lift=1.72  count=62360
  cereal                       lift=1.60  count=116167
  canned fruit applesauce      lift=1.58  count=30023
  cream                        lift=1.52  count=109941
  butter                       lift=1.52  count=89331
[paper goods] 인기도=0.063
  body lotions soap            lift=4.71  count=10500
  trash bags liners            lift=4.

## 5. 우리 상품끼리만 비교 (큰 장바구니 편향 확인)
lift 계산은 **전체 코너**로 하고, 추천 후보는 **우리 상품 코너 안에서만** 고른다.

전체 주문으로 계산하면 대량 장보기 주문 때문에 컵라면이 어디에나 추천된다(큰 장바구니 편향).
편의점처럼 작은 장바구니만 쓰면 이 쏠림이 줄어든다.

In [10]:
uniq = OURS.drop_duplicates("aisle_id")                    # 같은 코너(참치·스팸)는 한 번만
idx = [a2i[a] for a in uniq.aisle_id]
labels = list(uniq["name"])

def our_table(Cm, L, min_count):
    sub = pd.DataFrame(L[np.ix_(idx, idx)], index=labels, columns=labels).round(2)
    cnt = pd.DataFrame(Cm[np.ix_(idx, idx)], index=labels, columns=labels)
    return sub.where(cnt >= min_count)

summary = []
for title, (Cm, L, n, mc) in results.items():
    sub = our_table(Cm, L, mc)
    print(f"\n===== {title} =====")
    ramen = 0
    for k in labels:
        row = sub.loc[k].drop(k).dropna().sort_values(ascending=False).head(3)
        ramen += "컵라면" in row.index
        print(f"{k:6s} → " + ", ".join(f"{a}({v})" for a, v in row.items()))
    summary.append({"기준": title, "주문 수": n, "컵라면 쏠림(상위3 포함 상품 수)": ramen})
display(pd.DataFrame(summary))


===== 전체 주문 =====
참치캔    → 미니 티슈(1.87), 치약(1.74), 컵라면(1.74)
소형 우유팩 → 시리얼(1.6), 식빵(1.49), 요구르트(1.48)
비누 상자  → 치약(3.68), 미니 티슈(2.23), 도시락(1.5)
초콜릿 상자 → 쿠키(2.14), 견과류(1.81), 과자 봉지(1.72)
과자 봉지  → 쿠키(2.05), 크래커(1.84), 컵라면(1.78)
미니 티슈  → 치약(3.34), 비누 상자(2.23), 참치캔(1.87)
계란 모형  → 소시지(1.76), 식빵(1.62), 참치캔(1.6)
컵라면    → 소시지(2.08), 시리얼(1.98), 크래커(1.96)
콜라     → 쿠키(1.86), 미니 티슈(1.85), 초콜릿 상자(1.63)
생수     → 콜라(1.45), 미니 티슈(1.44), 비누 상자(1.43)
캔커피    → 미니 티슈(1.77), 치약(1.58), 초콜릿 상자(1.51)
요구르트   → 시리얼(1.51), 컵라면(1.49), 소형 우유팩(1.48)
식빵     → 컵라면(1.72), 시리얼(1.66), 소시지(1.62)
쿠키     → 크래커(2.17), 초콜릿 상자(2.14), 과자 봉지(2.05)
아이스크림  → 쿠키(1.82), 컵라면(1.64), 콜라(1.58)
시리얼    → 컵라면(1.98), 크래커(1.85), 쿠키(1.84)
견과류    → 초콜릿 상자(1.81), 크래커(1.64), 참치캔(1.48)
치약     → 비누 상자(3.68), 미니 티슈(3.34), 참치캔(1.74)
주스     → 컵라면(1.62), 쿠키(1.6), 콜라(1.59)
소시지    → 컵라면(2.08), 계란 모형(1.76), 참치캔(1.69)
크래커    → 쿠키(2.17), 컵라면(1.96), 시리얼(1.85)
도시락    → 컵라면(1.78), 쿠키(1.52), 식빵(1.5)

===== 코너 2~8개 =====
참치캔    → 소시지(1.25), 컵라면(1.21), 계란 모형(1.21

,기준,주문 수,컵라면 쏠림(상위3 포함 상품 수)
0,전체 주문,3214236,10
1,코너 2~8개,1965126,5
2,코너 2~5개,1167496,2


## 6. 최종 설정으로 저장
| 항목 | 값 |
|---|---|
| 사용 주문 | prior 중 **코너 2~5개** |
| 최소 동시 구매 | 200 |
| 최소 lift | 1.2 (그 아래는 우연 수준이라 추천 안 함) |
| 추천 짝이 없을 때 | 인기순(재고 있는 상품)으로 채움 |

저장 파일
- `rec_scores.json` : 모든 코너의 인기도 + 코너별 lift 상위 30개 (웹 서버용)
- `our_recs.json` : 우리 상품별 추천 목록 (바로 확인·데모용)
- `co_counts_small.npy`, `aisle_index.json` : 기준을 바꿔 다시 계산할 때 사용

In [11]:
C_use, L_use, N_use, _ = results["코너 2~5개"]
MIN_COUNT, MIN_LIFT, TOP_K = 200, 1.2, 30
cA = np.diag(C_use)

# 1) 코너 단위 점수 (웹 서버용)
rec = {"source": "Instacart prior orders", "n_orders": int(N_use), "basket_size": [2, 5],
       "min_count": MIN_COUNT, "min_lift": MIN_LIFT, "aisles": {}, "pairs": {}}
for i, a in enumerate(aisle_ids):
    rec["aisles"][aisle_name[a]] = {"id": int(a), "count": int(cA[i]),
                                    "popularity": round(float(cA[i] / N_use), 5)}
    cand = [(j, L_use[i, j]) for j in range(len(aisle_ids))
            if j != i and C_use[i, j] >= MIN_COUNT and L_use[i, j] >= MIN_LIFT]
    cand.sort(key=lambda x: -x[1])
    rec["pairs"][aisle_name[a]] = [{"aisle": aisle_name[aisle_ids[j]], "count": int(C_use[i, j]),
                                    "lift": round(float(l), 3)} for j, l in cand[:TOP_K]]

# 2) 우리 상품 단위 추천 (같은 코너 상품은 서로 대체 상품)
our = {}
for _, r in OURS.iterrows():
    i = a2i[r.aisle_id]
    cands, seen = [], set()
    for _, s in OURS.iterrows():
        if s.aisle_id == r.aisle_id or s.aisle_id in seen:   # 같은 코너 상품은 추천 목록에 한 번만
            continue
        j = a2i[s.aisle_id]
        if C_use[i, j] >= MIN_COUNT and L_use[i, j] >= MIN_LIFT:
            seen.add(s.aisle_id)
            cands.append({"id": s.id, "name": s["name"], "lift": round(float(L_use[i, j]), 3),
                          "count": int(C_use[i, j])})
    cands.sort(key=lambda x: -x["lift"])
    our[r.id] = {"name": r["name"], "aisle": r.aisle, "recommend": cands[:5],
                 "substitute": [s.id for _, s in OURS.iterrows() if s.aisle_id == r.aisle_id and s.id != r.id],
                 "popularity": round(float(cA[i] / N_use), 5)}

json.dump(rec, open(f"{OUT}/rec_scores.json", "w"), ensure_ascii=False, indent=1)
json.dump(our, open(f"{OUT}/our_recs.json", "w"), ensure_ascii=False, indent=1)
np.save(f"{OUT}/co_counts_small.npy", C_use)
json.dump([aisle_name[a] for a in aisle_ids], open(f"{OUT}/aisle_index.json", "w"))
pd.DataFrame(summary).to_csv(f"{OUT}/basket_size_comparison.csv", index=False)

for k, v in our.items():
    recs = ", ".join(f"{c['name']}({c['lift']})" for c in v["recommend"][:3]) or "(없음 → 인기순)"
    print(f"{v['name']:7s} → {recs}")
!ls -lh "{OUT}"

참치캔     → (없음 → 인기순)
스팸 통조림  → (없음 → 인기순)
소형 우유팩  → 시리얼(1.344), 요구르트(1.31)
비누 상자   → 치약(4.53), 미니 티슈(2.266)
초콜릿 상자  → 쿠키(2.025), 견과류(1.89), 과자 봉지(1.587)
과자 봉지   → 쿠키(1.838), 초콜릿 상자(1.587), 크래커(1.548)
미니 티슈   → 치약(3.885), 비누 상자(2.266), 콜라(1.364)
계란 모형   → 소시지(1.442), 식빵(1.248)
컵라면     → (없음 → 인기순)
콜라      → 생수(1.688), 초콜릿 상자(1.537), 쿠키(1.517)
생수      → 콜라(1.688), 미니 티슈(1.32), 캔커피(1.217)
캔커피     → 미니 티슈(1.23), 생수(1.217)
요구르트    → 소형 우유팩(1.31)
식빵      → 계란 모형(1.248)
쿠키      → 초콜릿 상자(2.025), 크래커(2.009), 과자 봉지(1.838)
아이스크림   → 쿠키(1.426), 콜라(1.204)
시리얼     → 소형 우유팩(1.344), 크래커(1.295), 쿠키(1.235)
견과류     → 초콜릿 상자(1.89), 크래커(1.262)
치약      → 비누 상자(4.53), 미니 티슈(3.885)
주스      → 콜라(1.351)
소시지     → 계란 모형(1.442)
크래커     → 쿠키(2.009), 과자 봉지(1.548), 초콜릿 상자(1.396)
도시락     → (없음 → 인기순)
total 259K
-rw------- 1 root root 2.5K Nov 26 06:24 aisle_index.json
-rw------- 1 root root  135 Nov 26 06:24 basket_size_comparison.csv
-rw------- 1 root root 137K Nov 26 06:24 co_counts_small.npy
-rw------- 1 root root

## 7. 다음 단계
1. **평가**: `order_products__train`(학습에 안 쓴 주문)으로 Hit@K 계산 → 무작위 / 인기순 / 동시 구매 / lift 비교
2. **item2vec**: 비교용 임베딩 모델 학습 → `aisle_emb.npy`
3. **웹 연동**: `products.json`에 aisle 추가, `rec_scores.json`으로 장바구니 추천
4. **밴딧**: 추천 노출·수락 기록으로 점수 보정 + 가상 손님 시뮬레이션